# Notebook 1 — Baseline Model: Traditional Credit Data Only

**Project:** Open Banking Credit Risk Model  
**Author:** Chetan Prakash | MSc FinTech, University of Glasgow  

---

## Objective
Establish the **baseline benchmark** — how well can a lender predict default using only traditional credit bureau features?

For new customers, only **external features** are available at onboarding:
- `checking_status`, `employment`, `housing`, `sex`, `age`, `credit_amount`, `existing_credits`

Internal bureau features (`credit_history`, `savings_status`) are **absent** — this is the thin-file problem.

## Data
| File | Role | Shape |
|------|------|-------|
| `old_cust_credit.csv` | Training | 9,000 rows × 11 cols |
| `new_cust_credit.csv` | Test | 1,000 rows × 8 cols |

## Models
- **Logistic Regression** — industry standard; interpretable; IFRS 9 aligned
- **XGBoost** — gradient-boosted ensemble; captures non-linear patterns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, roc_auc_score, brier_score_loss,
    confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve
)
import xgboost as xgb

SEED = 42
np.random.seed(SEED)
sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})
print('Libraries loaded successfully')

## 1. Load Data

In [ ]:
old_credit = pd.read_csv('../data/old_cust_credit.csv')
new_credit = pd.read_csv('../data/new_cust_credit.csv')

print('=== EXISTING CUSTOMERS (Training) ===')
print(f'  Shape        : {old_credit.shape}')
print(f'  Default rate : {old_credit["credit_defaults"].mean():.1%}')
print(f'  Columns      : {old_credit.columns.tolist()}')
print()
print('=== NEW CUSTOMERS (Test) ===')
print(f'  Shape        : {new_credit.shape}')
print(f'  Default rate : {new_credit["credit_defaults"].mean():.1%}')
print(f'  Columns      : {new_credit.columns.tolist()}')
print()
print('Missing columns for new customers (thin-file problem):')
missing = [c for c in old_credit.columns if c not in new_credit.columns]
print(f'  {missing}')

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, df, title in zip(axes,
    [old_credit, new_credit],
    ['Existing Customers — Training', 'New Customers — Test']):
    counts = df['credit_defaults'].value_counts().sort_index()
    ax.bar(['No Default', 'Default'], counts.values,
           color=['steelblue', 'tomato'], edgecolor='white', width=0.5)
    for i, v in enumerate(counts.values):
        ax.text(i, v + 20, str(v), ha='center', fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
plt.suptitle('Target Variable Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Preprocessing

In [ ]:
def encode_credit(df, is_train=True, ref_cols=None):
    """
    Encode credit features.
    Training uses all features; test uses external-only (thin-file scenario).
    """
    df = df.copy()
    df.drop(columns=['account_id'], errors='ignore', inplace=True)
    cats = (['checking_status','credit_history','employment','sex','housing','savings_status']
            if is_train else
            ['checking_status','employment','sex','housing'])
    df = pd.get_dummies(df, columns=[c for c in cats if c in df.columns], drop_first=True)
    y = df.pop('credit_defaults')
    X = df
    if ref_cols is not None:
        X = X.reindex(columns=ref_cols, fill_value=0)
    return X, y

X_train, y_train = encode_credit(old_credit, is_train=True)
X_test,  y_test  = encode_credit(new_credit, is_train=False, ref_cols=X_train.columns)

print(f'Training features : {X_train.shape}')
print(f'Test features     : {X_test.shape}')
print(f'\nFeatures zeroed for new customers (unavailable at onboarding):')
zeroed = [c for c in X_train.columns if X_test[c].sum() == 0]
print(f'  {zeroed[:8]} ...')

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train)

scaler = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_tr)
X_val_sc = scaler.transform(X_val)
X_te_sc  = scaler.transform(X_test)
print(f'\nTrain: {X_tr.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

## 3. Train Models

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Logistic Regression
lr = LogisticRegression(penalty='l2', solver='liblinear', max_iter=3000,
                         random_state=SEED, class_weight='balanced')
lr.fit(X_tr_sc, y_tr)
cv_lr = cross_val_score(lr, X_tr_sc, y_tr, cv=cv, scoring='roc_auc')
print(f'Logistic Regression CV AUC: {cv_lr.mean():.4f} +/- {cv_lr.std():.4f}')

# XGBoost
sp = (y_tr==0).sum() / (y_tr==1).sum()
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    scale_pos_weight=sp, eval_metric='logloss',
    random_state=SEED, tree_method='hist')
xgb_model.fit(X_tr_sc, y_tr, eval_set=[(X_val_sc, y_val)], verbose=False)
cv_xgb = cross_val_score(xgb_model, X_tr_sc, y_tr, cv=cv, scoring='roc_auc')
print(f'XGBoost CV AUC            : {cv_xgb.mean():.4f} +/- {cv_xgb.std():.4f}')

## 4. Evaluate on New Customers

In [ ]:
lr_pred  = lr.predict(X_te_sc);        lr_prob  = lr.predict_proba(X_te_sc)[:,1]
xgb_pred = xgb_model.predict(X_te_sc); xgb_prob = xgb_model.predict_proba(X_te_sc)[:,1]

for name, pred, prob in [('Logistic Regression', lr_pred, lr_prob),
                          ('XGBoost',            xgb_pred, xgb_prob)]:
    print(f'\n--- {name} ---')
    print(f'  AUC      : {roc_auc_score(y_test, prob):.4f}')
    print(f'  Accuracy : {accuracy_score(y_test, pred):.4f}')
    print(f'  Brier    : {brier_score_loss(y_test, prob):.4f}')
    print(classification_report(y_test, pred, target_names=['No Default','Default']))

In [ ]:
# ROC Curves + Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
for prob, label, color in [(lr_prob,'Logistic Regression','steelblue'),
                             (xgb_prob,'XGBoost','darkorange')]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax.plot(fpr, tpr, lw=2, color=color,
            label=f'{label} (AUC={roc_auc_score(y_test,prob):.3f})')
ax.plot([0,1],[0,1],'--',color='gray')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Baseline\n(Traditional Credit Only)', fontweight='bold')
ax.legend(loc='lower right')

for ax, pred, title, cmap in zip(axes[1:],
    [lr_pred, xgb_pred], ['Logistic Regression','XGBoost'], ['Blues','Oranges']):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
        display_labels=['No Default','Default']).plot(ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(f'{title}\nBaseline', fontweight='bold')

plt.suptitle('Notebook 1 — Baseline Model (Traditional Credit Data)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/01_baseline_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Results Summary

| Metric | Logistic Regression | XGBoost |
|--------|--------------------|---------|
| **Test AUC** | 0.7103 | 0.6804 |
| **Accuracy** | 69.9% | 67.4% |
| **Brier Score** | 0.1942 | 0.2042 |
| **Defaulters Caught** | 43% | 33% |

**Key Takeaway:** Without credit history or savings data, the model catches fewer than half of all defaulters.
This is the **thin-file problem** — the gap this project addresses using open banking transaction data.

---
**Next:** Notebook 2 tests whether transaction data alone can outperform this baseline.